# Network Slicing

## Prerequisites

Before starting, make sure you:

- Have completed the [Simple Scenario](./10_simple.ipynb) and [TDD](./20_tdd.ipynb) tutorials.
- Have a basic understanding of Python.
- You have a good understanding of the 5G architecture.
- You have a good understanding of network slicing.

## Experiment for this scenario

It the previous tutorials we have configured and deployed 5G networks. In this
tutorial we will configure network slices.

We will deploy a single 5G core with two slices, `internet` and `streaming`. One
UE will use the first slice and a second UE will be on the second slice. The
details of the slices are provided in the table below.

| Slice | SST | SD | DNN | IPv4 Network | Uplink AMBR | Downlink AMBR | QFI | 5QI | ARP Priority | DNNs |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| internet | 1 (eMBB) | `0xFFFFFF` | internet | `12.1.1.0/24` | 20 Mbps | 40 Mbps | 1 | 9 | 8 | {internet} |
| streaming | 1 (eMBB)| `0x100000` | streaming | `14.1.1.0/24` | 30 Mbps | 50 Mbps | 1 | 5 | 1 | {streaming} |

| UE | SUPI | S-NSSAI | DNN |
| --- | --- | --- | --- |
| `sim_ue1` | `001010000000100` | `internet` | `internet` |
| `sim_ue2` | `001010000000101` | `streaming` | `streaming` |


![Slicing topology](./figures/slicing.png)

The core network uses Open5GS and the RAN uses OAI. The radio channel between
the UEs and the gNB is simulated with `rfsim`.

## Metamodel and Deployement model definition

### Metamodel
In our previous tutorials, the metamodel remained simple as we leveraged its
default values. In this case as we are adding slices and DNNs, we have to
specify, for each network function, the slices and/or DNNs it refers too.

First we have to define the slices and name them, here we have the first slice
that we call `internet` and the second one that we name `streaming`. For each
function we will reference the slices via their friendly name.

For example, we specify that `upf1` is for the S-NSSAI named `internet` and our
framework will automatically know how to configure the UPF.

Slicing is configured on the UEs by setting their `subscriptions` attribute as
desired.

In [ ]:
from ifran import MetaModel

metamodel=MetaModel.from_yaml(
"""
slices:
  internet:
    sst: 1
    sd: 0xFFFFFF
    dnns:
      internet:
        ipv4Network: 12.1.1.0/24
        aggregateMaximumBitRate:
          uplink: 20Mbps
          downlink: 40Mbps
        qosFlows:
        - qfi: 1
          fiveQI: 9
          arp:
            priorityLevel: 8
  streaming:
    sst: 1
    sd: 0x100000
    dnns:
      streaming:
        ipv4Network: 14.1.1.0/24
        aggregateMaximumBitRate:
          uplink: 30Mbps
          downlink: 50Mbps
        qosFlows:
        - qfi: 1
          fiveQI: 5
          arp:
            priorityLevel: 1
core:
  components:
    upf:
      type: upf
      config:
        snssaiList:
          - internet
          - streaming
          
    nrf:
      type: nrf
      config:
        plmnList:
        - plmnId:
            mcc: '001'
            mnc: '01'
          snssaiList:
            - internet
            - streaming

    smf:
      type: smf
      config:
        snssaiList:
          - internet
          - streaming
        plmnList:
        - mcc: '001'
          mnc: '01'
        upfs:
          - upf

    pcf:
      type: pcf

    nssf:
      type: nssf
      config:
        nsiInfoList:
          - snssai: internet
          - snssai: streaming
        taInfoList:
        - tai:
            plmnId:
              mcc: '001'
              mnc: '01'
            tac: 1
          snssaiList:
            - internet
            - streaming
        amfInfoList:
        - amfList:
          - supportedNssaiAvailabilityData:
              tai:
                plmnId:
                  mcc: '001'
                  mnc: '01'
                tac: 1
              snssaiList:
                - internet
                - streaming
        plmnInfoList:
        - homePlmnId:
            mcc: '001'
            mnc: '01'

    udr:
      type: udr

    amf:
      type: amf
      config:
        guamiList:
          - plmnId:
              mcc: '001'
              mnc: '01'
        taiList:
        - plmnId:
            mcc: '001'
            mnc: '01'
          tac: 1
        plmnList:
        - plmnId:
            mcc: '001'
            mnc: '01'
          snssaiList:
            - internet
            - streaming

    udm:
      type: udm

    bsf:
      type: bsf

    ausf:
      type: ausf
      config:
        plmnList:
          - mcc: '001'
            mnc: '01'

ran:
  components:
    gnb:
      type: gnb
      config:
        plmnList:
          - plmnId:
              mcc: '001'
              mnc: '01'
            snssaiList:
              - internet
              - streaming
        amf: amf
ues:
  sim_ue1:
    config:
      supi: '001010000000100'
      subscriptions:
      - snssai: internet
        dnn: internet
      gnbs:
      - gnb
  sim_ue2:
    config:
      supi: '001010000000101'
      subscriptions:
      - snssai: streaming
        dnn: streaming
      gnbs:
      - gnb
""")

**NOTE:** as the radio channel is simulated we explicitly list the gNBs to which
the UEs are connected.

### Deployment model

There is no specific needs for the deployment model but for the sake of the
example we show how to specify the implementation for each 5G core function.

In [ ]:
from ifran import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
core:
  implementation: open5gs

ran:
  components:
    gnb:
      type: gnb
      implementation: oai
ues:
  sim_ue1:
    implementation: oai
    type: containerized
  sim_ue2:
    implementation: oai
    type: containerized
nodes:
  node-core:
    provisioner: my-provisionner
    roles:
    - core
    - ric
  node-ran:
    provisioner: my-provisionner
    roles:
    - ran
  node-ues:
    provisioner: my-provisionner
    roles:
    - ue
provisioners:
  my-provisionner:
    type: bi
    params:
      infra: be-gent1-bi-vm1
      project: post5g-beta
      experiment: slicing_scenario
      create_experiment: true
      experiment_duration: 1h
      node_duration: 1h
"""
)



## Provisioning the infrastructure

We provision the infrastructure as in the [Simple tutorial](./10_simple.ipynb)

In [ ]:
from ifran import ProvisioningManagement, K8SManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)
await prov.provision(quiet=False)

k8s = K8SManagement(preceding=prov)
await k8s.deploy(quiet=True)

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

TASK [provisioning/bi/provision : Select project 'post5g-beta' for 'node-core' provisioning] ***

TASK [provisioning/bi/provision : Select project 'post5g-beta' for 'node-ues' provisioning] ***

PLAY [localhost] ********************************

At this stage, the infrastructure is fully ready to support our 5G network
deployment.

But before deploying the 5G network, let's archive the metamodel and deployment
model resulting from this deployment.

In [4]:
from pathlib import Path
DIR="state/slicing"
Path(DIR).mkdir(parents=True, exist_ok=True)

In [5]:
prov.metamodel.to_yaml_file(f"{DIR}/metamodel.prov.yaml")
prov.deployment_model.to_yaml_file(f"{DIR}/deployment_model.prov.yaml")

k8s.metamodel.to_yaml_file(f"{DIR}/metamodel.k8s.yaml")
k8s.deployment_model.to_yaml_file(f"{DIR}/deployment_model.k8s.yaml")

## Deploy the 5G Network

### Deploy the 5G core

Now that the infrastructure is fully provisioned, we first have to deploy the 5G
core and check that it is composed of two UPFs.

In [ ]:
from ifran import CoreManagement
core = CoreManagement(preceding=k8s)
await core.deploy()


PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-database' chart] ***

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-bsf' chart] ******

TASK [helm/pr

In [ ]:
from ifran import ProvisioningManagement

prov = ProvisioningManagement(preceding=core)

print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "open5gs-core", "-o", "wide"],
    nodes=[prov.deployment_model.params.k8s.master_node],
    become=True, quiet=True, flatten=False))[prov.deployment_model.params.k8s.master_node])

NAME                            READY   STATUS    RESTARTS   AGE     IP              NODE        NOMINATED NODE   READINESS GATES
open5gs-amf-7b6dcfbfdf-8c7rw    1/1     Running   0          2m3s    10.244.134.7    node-core   <none>           <none>
open5gs-ausf-5778d7548b-l7497   1/1     Running   0          108s    10.244.134.8    node-core   <none>           <none>
open5gs-bsf-75cf9d7fb4-8wchp    1/1     Running   0          18s     10.244.134.14   node-core   <none>           <none>
open5gs-database-0              1/1     Running   0          2m45s   10.244.134.5    node-core   <none>           <none>
open5gs-nrf-65d98494d7-6k88m    1/1     Running   0          2m21s   10.244.134.6    node-core   <none>           <none>
open5gs-nssf-84cb8d64f7-rr9ws   1/1     Running   0          35s     10.244.134.13   node-core   <none>           <none>
open5gs-pcf-6f444b66c5-8p5ql    1/1     Running   0          80s     10.244.134.10   node-core   <none>           <none>
open5gs-smf-5c7c8d4c4c-

### Deploy the 5G RAN

We can now deploy the 5G RAN and verify that the gNB is deployed properly and 
supports both slices by analysing its logs.

In [ ]:
from ifran import RANManagement
ran = RANManagement(preceding=core)
await ran.deploy()

/workspaces/orchestrator/.venv-1/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='snssaiList', input_value=Slice(sst=1, sd=16777215,...maximumBitRate=None)])}), input_type=Slice])
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='snssaiList', input_value=Slice(sst=1, sd=1048576, ...maximumBitRate=None)])}), input_type=Slice])
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='ru', input_value=RU(type=<RUType.SIMULATED...xGainDB=35, rxGainDB=75), input_type=RU])
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='amf', input_value=IPv4Address('10.244.134.7'), input_type=IPv4Address])
  return self.__pydantic_serializer__.to_python(



PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-gnb' chart] **********
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node-core]

TASK [helm/prepare : Copy custom Helm chart directory for 'oai-gnb' if provided] ***
skipping: [node-core]

TASK [helm/prepare : Copy local built-in 'oai-gnb' Helm chart directory for 'oai-gnb'] ***
changed: [node-core]

TASK [helm/prepare : Render built-in Helm Chart.yaml templated file for 'oai-gnb'] ***
changed: [node-core]

TASK [helm/prepare : Render custom templated Helm files for 'oai-gnb' if provided] ***
skipping: [node-core]

TASK [helm/prepare : Render built-in templated Helm 

In [9]:
print(await ran.get_logs(components=["gnb"], quiet=True))

/proc/sys/kernel/core_pattern=|/usr/share/apport/apport -p%p -s%s -c%c -d%d -P%P -u%u -g%g -F%F -- %E
No configuration file /opt/oai-gnb/etc/gnb.conf found: attempting to find YAML config
== Configuration file:

Active_gNBs:
  - gnb
# Asn1_verbosity, choice in: none, info, annoying
Asn1_verbosity: none
gNBs:
  ##### Identification parameters:
  - gNB_ID: 0x1

    gNB_name: gnb

    tracking_area_code: 1

    plmn_list:
      - mcc: 001
        mnc: 01
        mnc_length: 2
        snssaiList:
          - sst: 1
            sd: 0xFFFFFF
          - sst: 1
            sd: 0x100000

    nr_cellid: 1

    ##### Physical parameters:
    min_rxtxtime: 6

    servingCellConfigCommon:
      #spCellConfigCommon
      - physCellId: 0
        #  downlinkConfigCommon
        #frequencyInfoDL
        absoluteFrequencySSB: 621312
        dl_absoluteFrequencyPointA: 620040
        #scs-SpecificCarrierList
        dl_offstToCarrier: 0
        # subcarrierSpacing
        # 0=kHz15, 1=kHz30, 2=kHz60, 3=

## Deploy the UEs

Now that we have a fully functional core and RAN, we can deploy the two UEs we
have defined in our metamodel. `sim_ue1` is in the slice `internet` and should
then obtain an IP address in the `12.1.1.0/24` prefix upon successful PDU
session establishement. As `sim_ue2` is in the slice `streaming`, it should get
an IP in the `14.1.1.0/24` prefix.


In [ ]:
from ifran import UEManagement

ues = UEManagement(preceding=ran)
await ues.deploy()


PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-sim-ue1' chart] ******

TASK [helm/prepare : Create Helm build directory for 'oai-sim-ue2' chart] ******
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node-core]

TASK [helm/prepare : Copy custom Helm chart directory for 'oai-sim-ue1' if provided] ***
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansib

In [11]:
logs = await ues.get_logs(components=["sim_ue1", "sim_ue2"], quiet=True, flatten=False)

In [12]:
print (logs["sim_ue1"]["nr-ue"])

CMDLINE: "/opt/oai-nr-ue/bin/nr-uesoftmodem" "-O" "/opt/oai-nr-ue/etc/nr-ue.conf" "-E" "--rfsim" 
[UTIL]   running in SA mode (no --phy-test, --do-ra, --nsa option present)
[UTIL]   threadCreate() for Tpool0_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool1_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool2_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool3_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool4_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool5_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool6_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool7_-1: creating thread with affinity ffffffff, priority 97
[OPT]    OPT disabled
[HW]     Version: Branch: develop-f80a88326801f5417405dea0b174879bac8940e0 Ab

In [13]:
print (logs["sim_ue2"]["nr-ue"])

CMDLINE: "/opt/oai-nr-ue/bin/nr-uesoftmodem" "-O" "/opt/oai-nr-ue/etc/nr-ue.conf" "-E" "--rfsim" 
[UTIL]   running in SA mode (no --phy-test, --do-ra, --nsa option present)
[UTIL]   threadCreate() for Tpool0_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool1_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool2_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool3_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool4_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool5_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool6_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool7_-1: creating thread with affinity ffffffff, priority 97
[OPT]    OPT disabled
[HW]     Version: Branch: develop-f80a88326801f5417405dea0b174879bac8940e0 Ab

In [14]:
command = ["bash", "-c", "'ip -4 addr show oaitun_ue1 | grep -oP \"(?<=inet )\\d+(\\.\\d+){3}\"'"]
result = (await ues.run_command(command=command, components=["sim_ue1"], quiet=True))
print (f"IP address of sim_ue1: {result}")

IP address of sim_ue1: 12.1.1.2



In [15]:
command = ["bash", "-c", "'ip -4 addr show oaitun_ue1 | grep -oP \"(?<=inet )\\d+(\\.\\d+){3}\"'"]
result = (await ues.run_command(command=command, components=["sim_ue2"], quiet=True))
print (f"IP address of sim_ue2: {result}")

IP address of sim_ue2: 14.1.1.2



We can also validate that the UEs use the correct slices by analysing their
logs.

In [16]:
print(await ues.get_logs(components=["sim_ue1"], quiet=True))

CMDLINE: "/opt/oai-nr-ue/bin/nr-uesoftmodem" "-O" "/opt/oai-nr-ue/etc/nr-ue.conf" "-E" "--rfsim" 
[UTIL]   running in SA mode (no --phy-test, --do-ra, --nsa option present)
[UTIL]   threadCreate() for Tpool0_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool1_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool2_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool3_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool4_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool5_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool6_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool7_-1: creating thread with affinity ffffffff, priority 97
[OPT]    OPT disabled
[HW]     Version: Branch: develop-f80a88326801f5417405dea0b174879bac8940e0 Ab

In [17]:
print(await ues.get_logs(components=["sim_ue2"], quiet=True))

CMDLINE: "/opt/oai-nr-ue/bin/nr-uesoftmodem" "-O" "/opt/oai-nr-ue/etc/nr-ue.conf" "-E" "--rfsim" 
[UTIL]   running in SA mode (no --phy-test, --do-ra, --nsa option present)
[UTIL]   threadCreate() for Tpool0_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool1_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool2_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool3_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool4_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool5_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool6_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool7_-1: creating thread with affinity ffffffff, priority 97
[OPT]    OPT disabled
[HW]     Version: Branch: develop-f80a88326801f5417405dea0b174879bac8940e0 Ab

Logs confirm that `sim_ue1` uses the `internet` slice.

```
[OIP]    TUN Interface oaitun_ue1 successfully configured, IPv4 12.1.1.2, IPv6 (null)
```

and that `sim_ue2` uses the `streaming` slice.

```
[OIP]    TUN Interface oaitun_ue1 successfully configured, IPv4 14.1.1.2, IPv6 (null)
```

To test that the UEs are well connected and that our network works as expected
we can run a ping from the UEs via their radio interace. In UERANSIM, the
interface is called `uesimtun0`.

To run commands directly on the nodes,

In [18]:
command = ["ping", "-I", "oaitun_ue1", "-c", 10, "-D", "8.8.8.8"]
pings = await ues.run_command(command=command, components=["sim_ue1", "sim_ue2"], quiet=True, flatten=False)

ping from `sim_ue1`:

In [19]:
print (pings["sim_ue1"])

PING 8.8.8.8 (8.8.8.8) from 12.1.1.2 oaitun_ue1: 56(84) bytes of data.
[1793813154.655128] 64 bytes from 8.8.8.8: icmp_seq=1 ttl=117 time=42.4 ms
[1793813155.655149] 64 bytes from 8.8.8.8: icmp_seq=2 ttl=117 time=41.0 ms
[1793813156.649202] 64 bytes from 8.8.8.8: icmp_seq=3 ttl=117 time=34.0 ms
[1793813157.663930] 64 bytes from 8.8.8.8: icmp_seq=4 ttl=117 time=47.8 ms
[1793813158.655990] 64 bytes from 8.8.8.8: icmp_seq=5 ttl=117 time=37.9 ms
[1793813159.670139] 64 bytes from 8.8.8.8: icmp_seq=6 ttl=117 time=51.1 ms
[1793813160.668864] 64 bytes from 8.8.8.8: icmp_seq=7 ttl=117 time=48.7 ms
[1793813161.664605] 64 bytes from 8.8.8.8: icmp_seq=8 ttl=117 time=43.1 ms
[1793813162.688028] 64 bytes from 8.8.8.8: icmp_seq=9 ttl=117 time=65.4 ms
[1793813163.669688] 64 bytes from 8.8.8.8: icmp_seq=10 ttl=117 time=46.5 ms

--- 8.8.8.8 ping statistics ---
10 packets transmitted, 10 received, 0% packet loss, time 9010ms
rtt min/avg/max/mdev = 33.993/45.775/65.363/8.165 ms



ping from `sim_ue2`:

In [20]:
print (pings["sim_ue2"])

PING 8.8.8.8 (8.8.8.8) from 14.1.1.2 oaitun_ue1: 56(84) bytes of data.
[1793813154.725194] 64 bytes from 8.8.8.8: icmp_seq=1 ttl=117 time=37.5 ms
[1793813155.731636] 64 bytes from 8.8.8.8: icmp_seq=2 ttl=117 time=42.4 ms
[1793813156.717848] 64 bytes from 8.8.8.8: icmp_seq=3 ttl=117 time=27.1 ms
[1793813157.733637] 64 bytes from 8.8.8.8: icmp_seq=4 ttl=117 time=42.5 ms
[1793813158.728415] 64 bytes from 8.8.8.8: icmp_seq=5 ttl=117 time=35.6 ms
[1793813159.745492] 64 bytes from 8.8.8.8: icmp_seq=6 ttl=117 time=50.9 ms
[1793813160.744435] 64 bytes from 8.8.8.8: icmp_seq=7 ttl=117 time=47.9 ms
[1793813161.748414] 64 bytes from 8.8.8.8: icmp_seq=8 ttl=117 time=49.9 ms
[1793813162.728765] 64 bytes from 8.8.8.8: icmp_seq=9 ttl=117 time=29.6 ms
[1793813163.744750] 64 bytes from 8.8.8.8: icmp_seq=10 ttl=117 time=43.9 ms

--- 8.8.8.8 ping statistics ---
10 packets transmitted, 10 received, 0% packet loss, time 9013ms
rtt min/avg/max/mdev = 27.089/40.736/50.877/7.750 ms



Finally, we can archive the metamodel and deployment model in file for keeping
track of the experiment models.

In [21]:
ues.metamodel.to_yaml_file(f"{DIR}/metamodel.ues.yaml")
ues.deployment_model.to_yaml_file(f"{DIR}/deployment_model.ues.yaml")